# Cross-Sell Oriented VAE Upgrade Walkthrough

This notebook documents and demonstrates the VAE changes made for cross-selling use cases.

## What was added
- Binary cross-sell target construction from product-group share features.
- Cross-sell training mode with BCE-with-logits auxiliary supervision.
- Optional sampled BPR pairwise ranking loss for top-K behavior.
- Built-in top-K ranking metrics: Recall@K, NDCG@K, MAP@K, and revenue-weighted recall@K.
- Extended config knobs for cross-sell weighting and ranking behavior.

## 1) Imports
The core functionality is now exposed from `embeddings` package exports.

In [ ]:
import numpy as np
import pandas as pd

from embeddings import (
    TORCH_AVAILABLE,
    RecommendationAwareVAEConfig,
    build_recommendation_targets,
    evaluate_cross_sell_predictions,
    preprocess_feature_matrix,
    train_recommendation_aware_vae,
)

print('Torch available:', TORCH_AVAILABLE)

## 2) Create a synthetic feature frame
This simulates customer features with `share__Product group__...` columns used to build cross-sell targets.

In [ ]:
rng = np.random.default_rng(42)
n_customers = 1200
n_product_groups = 12

feature_df = pd.DataFrame({
    'recency_days': rng.gamma(2.0, 12.0, size=n_customers),
    'frequency_90d': rng.poisson(5.5, size=n_customers).astype(float),
    'monetary_90d': rng.lognormal(mean=4.0, sigma=0.6, size=n_customers),
})

for i in range(n_product_groups):
    probs = rng.uniform(0.08, 0.35, size=n_customers)
    shares = (rng.random(size=n_customers) < probs).astype(float) * rng.uniform(0.05, 0.6, size=n_customers)
    feature_df[f'share__Product group__{i:02d}'] = shares

feature_df.head()

## 3) Build training matrix and generated targets
`build_recommendation_targets` now includes `cross_sell_product_group_binary` as a binary target matrix.

In [ ]:
cleaned_df, matrix, imputer, scaler = preprocess_feature_matrix(feature_df)
targets = build_recommendation_targets(cleaned_df)

print('Matrix shape:', matrix.shape)
print('Available target keys:', sorted(targets.keys()))
print('Cross-sell target shape:', targets['cross_sell_product_group_binary'].shape)

## 4) Configure cross-sell VAE training
Key options:
- `loss_variant='cross_sell'` enables BCE-based auxiliary learning.
- `cross_sell_bpr_weight` adds optional sampled BPR ranking pressure.
- `cross_sell_k_values` controls reported ranking metrics.

In [ ]:
config = RecommendationAwareVAEConfig(
    loss_variant='cross_sell',
    latent_dim=24,
    hidden_dims=(512, 256, 128),
    beta=0.08,
    kl_warmup_epochs=20,
    auxiliary_weight=1.0,
    cross_sell_target_key='cross_sell_product_group_binary',
    cross_sell_bce_weight=1.0,
    cross_sell_bpr_weight=0.15,
    cross_sell_negative_samples=4,
    cross_sell_k_values=(5, 10, 20),
    epochs=30,
    batch_size=256,
    learning_rate=1e-3,
    random_state=42,
)
config

## 5) Train and inspect losses
If Torch is available, this cell trains the recommendation-aware VAE in cross-sell mode.

In [ ]:
if not TORCH_AVAILABLE:
    print('Torch is not installed in this environment. Install torch to run training.')
else:
    result = train_recommendation_aware_vae(
        matrix,
        feature_df=cleaned_df,
        config=config,
    )
    print('Latent shape:', result.latent_mean.shape)
    display(result.history.tail(5))

## 6) Cross-sell ranking metrics
The new training path can return a metrics table directly from model logits and binary targets.

In [ ]:
if TORCH_AVAILABLE:
    if result.metrics is not None and not result.metrics.empty:
        display(result.metrics)
    else:
        print('No built-in metrics returned. Recomputing from saved predictions...')
        scores = result.auxiliary_predictions['cross_sell_product_group_binary']
        y_true = targets['cross_sell_product_group_binary']
        metrics = evaluate_cross_sell_predictions(scores, y_true, k_values=(5, 10, 20))
        display(metrics)

## 7) Notes for production cross-sell usage
- Replace the proxy binary target with a true future-window label (temporal split).
- Keep BCE head weight high enough to shape ranking quality.
- Use BPR weight carefully (start small, e.g. 0.05 to 0.2).
- Validate with Recall@K and NDCG@K on strictly future interactions.